# 01 - Explore & quality-check openFDA drug/label (bronze)

Feeds the retrieval extension (BR-16, BR-17, TR-50 ... TR-57). Every figure the build
plan currently carries is a **sample**; this notebook replaces them with a full pass.

Eight questions, each feeding a decision downstream:

| # | Question | Feeds |
|---|---|---|
| 1 | Structure - what fields, and how nested? | the chunker |
| 2 | Coverage - how often is `openfda` populated? | corpus scope |
| 3 | Section lengths - percentiles | chunk size |
| 4 | Duplication - how much label text is repeated? | dedup policy |
| 5 | Identity - `set_id` vs `id` vs `product_ndc` | the stable chunk key |
| 6 | Scope - prescription vs OTC | corpus scope |
| 7 | Joinability - can a label reach the marts? | the query-time filter |
| 8 | Scope revisited - was excluding OTC right? | corpus scope, again |

**Engine: DuckDB, not Spark.** `rag/README.md` commits this package to no Snowflake,
no Airflow and no Spark, so nothing here needs a container. The corpus is 8.5 GB of
newline-delimited JSON, which streams fine on one laptop.

In [1]:
import glob, hashlib, json, os, re, time
from collections import Counter

import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

BRONZE   = "D:/capstone/data/bronze/drug_label"
OFFLINE  = "D:/capstone/data/offline"
RAGDATA  = "D:/capstone/data/rag"
SECTIONS = f"{RAGDATA}/labels_sections.parquet"   # one row per (record, text section)
META     = f"{RAGDATA}/labels_meta.parquet"       # one row per record

PARTS = sorted(glob.glob(f"{BRONZE}/part-*.json"))
total_bytes = sum(os.path.getsize(p) for p in PARTS)
print(f"{len(PARTS)} part files, {total_bytes:,} bytes "
      f"({total_bytes/1e9:.2f} GB decimal / {total_bytes/2**30:.2f} GiB)")

14 part files, 8,549,338,919 bytes (8.55 GB decimal / 7.96 GiB)


### Why flatten to Parquet first?

Same reason the drug/event notebook cached to Parquet: re-reading the raw JSON for every
question is the slow part, and in a notebook you ask a lot of questions.

Here it matters more. The corpus is 8.5 GB of JSONL with deeply nested, wildly varying
records, so a single pass costs minutes. Flattening once into two narrow Parquet files
turns every question below into a millisecond SQL query.

Two outputs, because the grain differs:

- **`labels_sections.parquet`** - one row per *(record, text section)*, carrying
  `n_chars` and a hash of the text. This is the grain for length percentiles and for
  duplication.
- **`labels_meta.parquet`** - one row per *record*, carrying identity and the `openfda`
  metadata. This is the grain for coverage, scope and joinability.

The text itself is **not** stored. This pass measures shape, not content, and keeping the
prose out holds these files to a few hundred MB.

The cell below skips the work if the Parquet already exists. Set `FORCE = True` to rebuild.

In [2]:
FORCE = False
BATCH = 20_000          # records per Parquet row-group flush, keeps memory flat

SEC_SCHEMA = pa.schema([
    ("set_id",    pa.string()),
    ("section",   pa.string()),
    ("n_chars",   pa.int32()),
    ("text_hash", pa.binary(8)),
])
META_SCHEMA = pa.schema([
    ("set_id",            pa.string()),
    ("doc_id",            pa.string()),
    ("version",           pa.string()),
    ("effective_time",    pa.string()),
    ("has_openfda",       pa.bool_()),
    ("product_type",      pa.string()),
    ("brand_name",        pa.string()),
    ("generic_name",      pa.string()),
    ("manufacturer_name", pa.string()),
    ("product_ndc",       pa.string()),
    ("rxcui",             pa.string()),
    ("route",             pa.string()),
    ("n_sections",        pa.int32()),
])

def first(d, key):
    v = d.get(key)
    return str(v[0]) if isinstance(v, list) and v else None

def text_of(v):
    "openFDA text sections are lists of strings; everything else is not text."
    if isinstance(v, list) and v and all(isinstance(x, str) for x in v):
        return "\n".join(v)
    return None

if os.path.exists(SECTIONS) and os.path.exists(META) and not FORCE:
    print("Parquet already built. Set FORCE = True to rebuild.")
else:
    os.makedirs(RAGDATA, exist_ok=True)
    t0, n = time.time(), 0
    sec_buf = {k: [] for k in SEC_SCHEMA.names}
    met_buf = {k: [] for k in META_SCHEMA.names}
    sw = pq.ParquetWriter(SECTIONS, SEC_SCHEMA, compression="zstd")
    mw = pq.ParquetWriter(META,     META_SCHEMA, compression="zstd")

    def flush():
        if sec_buf["set_id"]:
            sw.write_table(pa.table(sec_buf, schema=SEC_SCHEMA))
        if met_buf["set_id"]:
            mw.write_table(pa.table(met_buf, schema=META_SCHEMA))
        for b in (sec_buf, met_buf):
            for k in b:
                b[k].clear()

    for path in PARTS:
        with open(path, "r", encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                n += 1
                sid = rec.get("set_id")

                n_sec = 0
                for key, val in rec.items():
                    txt = text_of(val)
                    if txt is None:
                        continue
                    n_sec += 1
                    sec_buf["set_id"].append(sid)
                    sec_buf["section"].append(key)
                    sec_buf["n_chars"].append(len(txt))
                    sec_buf["text_hash"].append(
                        hashlib.blake2b(txt.encode("utf-8"), digest_size=8).digest())

                o = rec.get("openfda") or {}
                met_buf["set_id"].append(sid)
                met_buf["doc_id"].append(rec.get("id"))
                met_buf["version"].append(str(rec.get("version")))
                met_buf["effective_time"].append(rec.get("effective_time"))
                met_buf["has_openfda"].append(bool(o))
                met_buf["product_type"].append(first(o, "product_type"))
                met_buf["brand_name"].append(first(o, "brand_name"))
                met_buf["generic_name"].append(first(o, "generic_name"))
                met_buf["manufacturer_name"].append(first(o, "manufacturer_name"))
                met_buf["product_ndc"].append(first(o, "product_ndc"))
                met_buf["rxcui"].append(first(o, "rxcui"))
                met_buf["route"].append(first(o, "route"))
                met_buf["n_sections"].append(n_sec)

                if n % BATCH == 0:
                    flush()

        print(f"  {os.path.basename(path)}  cumulative {n:,} records "
              f"({time.time()-t0:.0f}s)", flush=True)

    flush()
    sw.close()
    mw.close()
    print(f"\nDone. {n:,} records in {time.time()-t0:.0f}s")
    print(f"  {SECTIONS}  {os.path.getsize(SECTIONS)/1e6:.0f} MB")
    print(f"  {META}      {os.path.getsize(META)/1e6:.0f} MB")

  part-0000.json  cumulative 20,000 records (6s)
  part-0001.json  cumulative 40,000 records (12s)
  part-0002.json  cumulative 60,000 records (17s)
  part-0003.json  cumulative 80,000 records (22s)
  part-0004.json  cumulative 100,000 records (26s)
  part-0005.json  cumulative 120,000 records (31s)
  part-0006.json  cumulative 140,000 records (34s)
  part-0007.json  cumulative 160,000 records (38s)
  part-0008.json  cumulative 180,000 records (41s)
  part-0009.json  cumulative 200,000 records (44s)
  part-0010.json  cumulative 220,000 records (49s)
  part-0011.json  cumulative 240,000 records (55s)
  part-0012.json  cumulative 260,000 records (61s)
  part-0013.json  cumulative 261,258 records (61s)

Done. 261,258 records in 61s
  D:/capstone/data/rag/labels_sections.parquet  48 MB
  D:/capstone/data/rag/labels_meta.parquet      14 MB


In [3]:
con = duckdb.connect()
con.execute(f"CREATE VIEW sections AS SELECT * FROM '{SECTIONS}'")
con.execute(f"CREATE VIEW meta     AS SELECT * FROM '{META}'")

N_RECORDS = con.sql("SELECT count(*) FROM meta").fetchone()[0]
N_SECTION_ROWS = con.sql("SELECT count(*) FROM sections").fetchone()[0]
print(f"records      {N_RECORDS:,}")
print(f"section rows {N_SECTION_ROWS:,}   "
      f"({N_SECTION_ROWS/N_RECORDS:.1f} text sections per record)")

records      261,258
section rows 4,442,377   (17.0 text sections per record)


## 1. Structure - what fields, and how nested?  ->  feeds the chunker

openFDA label records are one JSON object per line. Nearly every field is a **list of
strings** rather than a string, and the field set varies enormously by record, because a
prescription label and an OTC monograph label carry different SPL sections.

The chunker has to know which sections exist and how often, so it can allowlist the ones
carrying safety content rather than embedding whatever happens to be present.

### Schema - what the records actually look like

The drug/event notebook used `df.printSchema()` here. There is no Spark in this package, so
the equivalent is derived directly: walk every record, record each field path, its JSON type,
and **how often it is present**.

That last column is the important difference. In drug/event nearly every field appeared on
every record, so a bare type tree was enough. Here the field set varies enormously between a
prescription label and an OTC monograph, so a type without a presence figure is misleading.

Two things to watch for in the output:

- **What type is nearly everything?** If it matches drug/event's "every field is a string"
  finding, that is one less thing for the chunker to handle.
- **`openfda` presence.** Compare the presence of `openfda` itself against the presence of
  its subfields such as `openfda.brand_name`. If they differ, the obvious way to test for
  metadata is wrong, and the code in section 2 has to test something else.

This is a second full pass over the raw JSON (about 40 seconds). It reads the raw records
rather than the flattened Parquet, because the Parquet deliberately dropped structure.

In [17]:
from collections import Counter, defaultdict

def json_type(v):
    """The JSON type of a value, describing list contents rather than just 'array'."""
    if v is None:              return "null"
    if isinstance(v, bool):    return "boolean"
    if isinstance(v, (int, float)): return "number"
    if isinstance(v, str):     return "string"
    if isinstance(v, dict):    return "struct"
    if isinstance(v, list):
        inner = {json_type(x) for x in v[:5]} or {"empty"}
        return f"array<{'|'.join(sorted(inner))}>"
    return type(v).__name__

def walk(obj, present, types, prefix=""):
    """Record every field path in one record, descending into nested objects."""
    for k, v in obj.items():
        path = f"{prefix}{k}"
        present[path] += 1
        types[path][json_type(v)] += 1
        if isinstance(v, dict):
            walk(v, present, types, path + ".")

present, types, n = Counter(), defaultdict(Counter), 0
t0 = time.time()
for path in PARTS:
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            n += 1
            walk(rec, present, types)

print(f"{n:,} records, {len(present):,} distinct field paths, {time.time()-t0:.0f}s\n")

# Render as a printSchema-style tree, with presence added.
lines = ["root"]
for p in sorted(present, key=lambda k: (k.split(".")[0] != "openfda", -present[k], k)):
    depth = p.count(".")
    name  = p.split(".")[-1]
    t     = types[p].most_common(1)[0][0]
    indent = " |   " * depth
    lines.append(f" |--{indent} {name}: {t}  ({100*present[p]/n:.1f}%)")
SCHEMA_TREE = "\n".join(lines)
print(SCHEMA_TREE[:4000])
print(f"\n... {len(lines)-1} field paths total")

261,258 records, 183 distinct field paths, 44s

root
 |-- openfda: struct  (100.0%)
 |-- |    brand_name: array<string>  (33.1%)
 |-- |    generic_name: array<string>  (33.1%)
 |-- |    manufacturer_name: array<string>  (33.1%)
 |-- |    package_ndc: array<string>  (33.1%)
 |-- |    product_ndc: array<string>  (33.1%)
 |-- |    product_type: array<string>  (33.1%)
 |-- |    spl_id: array<string>  (33.1%)
 |-- |    spl_set_id: array<string>  (33.1%)
 |-- |    route: array<string>  (32.6%)
 |-- |    substance_name: array<string>  (32.4%)
 |-- |    unii: array<string>  (32.4%)
 |-- |    application_number: array<string>  (28.6%)
 |-- |    is_original_packager: array<boolean>  (24.8%)
 |-- |    rxcui: array<string>  (24.5%)
 |-- |    upc: array<string>  (10.5%)
 |-- |    nui: array<string>  (10.1%)
 |-- |    pharm_class_epc: array<string>  (9.5%)
 |-- |    original_packager_product_ndc: array<string>  (8.3%)
 |-- |    pharm_class_cs: array<string>  (5.5%)
 |-- |    pharm_class_pe: array<st

In [18]:
# The same thing as a table, sorted by presence, and written to a schema document.
print(f"{'field path':<58}{'type':<20}{'present':>10}{'pct':>8}")
print("-" * 96)
for p in sorted(present, key=lambda k: (-present[k], k))[:60]:
    t = types[p].most_common(1)[0][0]
    print(f"{p:<58}{t:<20}{present[p]:>10,}{100*present[p]/n:>7.1f}%")

rows = "\n".join(
    f"| `{p}` | `{types[p].most_common(1)[0][0]}` | {present[p]:,} | {100*present[p]/n:.1f}% |"
    for p in sorted(present, key=lambda k: (-present[k], k))
)
doc = f"""# drug/label - actual data schema (derived from the records)

The real structure of the ingested **bronze `drug_label`** data ({n:,} records).
Regenerate any time by re-running the schema cell in
`rag/notebooks/01_explore_drug_label.ipynb`.

## Schema

```
{SCHEMA_TREE}
```

## All field paths by presence

| field path | type | present | % of records |
|---|---|---|---|
{rows}
"""
out = "../notes/drug_label_schema.md"
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, "w", encoding="utf-8") as fh:
    fh.write(doc)
print(f"\nwritten: {out}  ({len(doc):,} chars)")

field path                                                type                   present     pct
------------------------------------------------------------------------------------------------
effective_time                                            string                 261,258  100.0%
id                                                        string                 261,258  100.0%
openfda                                                   struct                 261,258  100.0%
set_id                                                    string                 261,258  100.0%
version                                                   string                 261,258  100.0%
spl_product_data_elements                                 array<string>          260,960   99.9%
package_label_principal_display_panel                     array<string>          260,898   99.9%
indications_and_usage                                     array<string>          251,968   96.4%
dosage_and_administration     

### Schema - findings

*TODO after running:*

- How many distinct field paths exist, and how does that compare to drug/event?
- Is every field the same type, as in drug/event where everything was a string? Which field
  is the exception?
- **`openfda` presence vs `openfda.*` subfield presence:** are they the same number? If not,
  what does that mean for how section 2 has to test for metadata, and what would go wrong
  with the naive test?
- How deep does the nesting go? Does it need exploding the way `patient.drug[]` did, or is
  it flat enough to read directly?
- How long is the tail of rare fields, and does anything in it belong in the allowlist?

In [4]:
con.sql(f'''
    SELECT section,
           count(*)                                   AS present,
           round(100.0*count(*)/{N_RECORDS}, 1)       AS pct_of_records
    FROM sections
    GROUP BY 1
    ORDER BY present DESC
    LIMIT 40
''').show(max_rows=40)

┌────────────────────────────────────────────────────────────┬─────────┬────────────────┐
│                          section                           │ present │ pct_of_records │
│                          varchar                           │  int64  │     double     │
├────────────────────────────────────────────────────────────┼─────────┼────────────────┤
│ spl_product_data_elements                                  │  260960 │           99.9 │
│ package_label_principal_display_panel                      │  260898 │           99.9 │
│ indications_and_usage                                      │  251968 │           96.4 │
│ dosage_and_administration                                  │  251405 │           96.2 │
│ warnings                                                   │  207792 │           79.5 │
│ inactive_ingredient                                        │  166004 │           63.5 │
│ purpose                                                    │  163349 │           62.5 │
│ keep_out

In [5]:
# The scalar (non-text) fields, for completeness: these are the identity carriers.
con.sql('''
    SELECT count(*)                                   AS records,
           count(set_id)                              AS has_set_id,
           count(doc_id)                              AS has_id,
           count(effective_time)                      AS has_effective_time,
           count(*) FILTER (WHERE has_openfda)        AS has_openfda
    FROM meta
''').show()

┌─────────┬────────────┬────────┬────────────────────┬─────────────┐
│ records │ has_set_id │ has_id │ has_effective_time │ has_openfda │
│  int64  │   int64    │ int64  │       int64        │    int64    │
├─────────┼────────────┼────────┼────────────────────┼─────────────┤
│  261258 │     261258 │ 261258 │             261258 │       86367 │
└─────────┴────────────┴────────┴────────────────────┴─────────────┘



### Structure - findings

*TODO after running:*

- How many distinct text sections appear across the corpus?
- Which sections are near-universal, and which are the long tail?
- Do the OTC monograph fields (`purpose`, `stop_use`, `do_not_use`, `when_using`,
  `keep_out_of_reach_of_children`) dominate the presence list? If so, that is the first
  hint that most of this corpus is not prescription drugs.
- Which sections belong in the Phase 1 allowlist?

## 2. Coverage - how often is `openfda` populated?  ->  feeds corpus scope

The `openfda` block carries brand name, generic name, manufacturer, NDC and RxCUI. openFDA
only fills it when the SPL could be harmonised against the NDC directory.

This is the single most important number in the notebook. **A record with an empty
`openfda` block cannot be attributed to a drug, cannot be cited usefully, and cannot be
joined to the marts.** If coverage is low, the v1 corpus shrinks by that factor, and that
is a principled scope filter rather than an arbitrary one.

The plan was drafted assuming roughly 41%, from a strided sample of one part file.

In [6]:
con.sql(f'''
    SELECT
        round(100.0*count(*) FILTER (WHERE has_openfda)      /{N_RECORDS}, 1) AS pct_openfda,
        round(100.0*count(product_ndc)                       /{N_RECORDS}, 1) AS pct_product_ndc,
        round(100.0*count(rxcui)                             /{N_RECORDS}, 1) AS pct_rxcui,
        round(100.0*count(generic_name)                      /{N_RECORDS}, 1) AS pct_generic_name,
        round(100.0*count(brand_name)                        /{N_RECORDS}, 1) AS pct_brand_name,
        round(100.0*count(manufacturer_name)                 /{N_RECORDS}, 1) AS pct_manufacturer,
        round(100.0*count(product_type)                      /{N_RECORDS}, 1) AS pct_product_type
    FROM meta
''').show()

┌─────────────┬─────────────────┬───────────┬──────────────────┬────────────────┬──────────────────┬──────────────────┐
│ pct_openfda │ pct_product_ndc │ pct_rxcui │ pct_generic_name │ pct_brand_name │ pct_manufacturer │ pct_product_type │
│   double    │     double      │  double   │      double      │     double     │      double      │      double      │
├─────────────┼─────────────────┼───────────┼──────────────────┼────────────────┼──────────────────┼──────────────────┤
│        33.1 │            33.1 │      24.5 │             33.1 │           33.1 │             33.1 │             33.1 │
└─────────────┴─────────────────┴───────────┴──────────────────┴────────────────┴──────────────────┴──────────────────┘



### Coverage - findings

*TODO after running:*

- Measured `openfda` coverage: ____%. Does it confirm or contradict the sampled ~41%?
- How many records survive the "attributable" filter? `261,258 x coverage = ____`
- `rxcui` coverage matters separately: it is the cleanest join key to the marts, with no
  typo or salt-stripping surface. What fraction of labels carry one?

## 3. Section lengths - percentiles  ->  feeds chunk size

TR-50 specified "one chunk per SPL section", on the reasoning that a label section is the
natural retrieval unit. That holds only if sections fit inside an embedding model's
context window.

Percentiles, not averages: the distribution has a very long tail, and an average would
hide exactly the sections that break a chunker. p50 is the median, p90 means 9 in 10 are
shorter, p99 means 99 in 100 are shorter.

At roughly 4 characters per token, a 512-token model window is about **2,048 characters**.

In [7]:
con.sql(f'''
    SELECT section,
           count(*)                              AS present,
           round(100.0*count(*)/{N_RECORDS}, 1)  AS pct,
           quantile_cont(n_chars, 0.50)::INT     AS p50,
           quantile_cont(n_chars, 0.90)::INT     AS p90,
           quantile_cont(n_chars, 0.99)::INT     AS p99,
           max(n_chars)                          AS max_chars
    FROM sections
    GROUP BY 1
    ORDER BY present DESC
    LIMIT 30
''').show(max_rows=30)

┌────────────────────────────────────────────────────────────┬─────────┬────────┬───────┬───────┬───────┬───────────┐
│                          section                           │ present │  pct   │  p50  │  p90  │  p99  │ max_chars │
│                          varchar                           │  int64  │ double │ int32 │ int32 │ int32 │   int32   │
├────────────────────────────────────────────────────────────┼─────────┼────────┼───────┼───────┼───────┼───────────┤
│ spl_product_data_elements                                  │  260960 │   99.9 │   251 │   676 │  1702 │     82333 │
│ package_label_principal_display_panel                      │  260898 │   99.9 │    91 │   623 │  2429 │     38569 │
│ indications_and_usage                                      │  251968 │   96.4 │   173 │  1759 │  5214 │     77114 │
│ dosage_and_administration                                  │  251405 │   96.2 │   351 │  5108 │ 13078 │     56151 │
│ warnings                                              

In [8]:
# The question that actually decides the chunker: what share of each safety-critical
# section OVERFLOWS a single 512-token chunk?
ALLOWLIST = ('boxed_warning','adverse_reactions','warnings','warnings_and_cautions',
             'contraindications','drug_interactions','indications_and_usage',
             'use_in_specific_populations','dosage_and_administration','overdosage')
CAP_CHARS = 512 * 4

con.sql(f'''
    SELECT section,
           count(*)                                                          AS present,
           round(100.0*count(*) FILTER (WHERE n_chars > {CAP_CHARS})/count(*), 1)
                                                                             AS pct_overflow,
           round(avg(n_chars)/{CAP_CHARS}, 2)                                AS mean_chunks_needed
    FROM sections
    WHERE section IN {ALLOWLIST}
    GROUP BY 1
    ORDER BY pct_overflow DESC
''').show(max_rows=20)

┌─────────────────────────────┬─────────┬──────────────┬────────────────────┐
│           section           │ present │ pct_overflow │ mean_chunks_needed │
│           varchar           │  int64  │    double    │       double       │
├─────────────────────────────┼─────────┼──────────────┼────────────────────┤
│ use_in_specific_populations │   43159 │         94.6 │                3.4 │
│ warnings_and_cautions       │   46832 │         83.5 │               4.61 │
│ adverse_reactions           │   91406 │         70.6 │               2.87 │
│ drug_interactions           │   68390 │         56.3 │               1.71 │
│ boxed_warning               │   33009 │         22.9 │               0.79 │
│ dosage_and_administration   │  251405 │         22.7 │               0.78 │
│ warnings                    │  207792 │         16.4 │               0.66 │
│ overdosage                  │   83888 │         16.1 │               0.61 │
│ indications_and_usage       │  251968 │          8.7 │        

### Section lengths - findings

*TODO after running:*

- Which sections overflow a 512-token window, and by how much?
- Is it true that the safety-critical sections are the ones that overflow, while the
  trivial OTC ones (`purpose`, `questions`) are far too short to embed usefully?
- Verdict on TR-50: does "one chunk per section" survive?
- What minimum chunk length should Phase 1 enforce at the short end?
- Confirm the `*_table` sections are as extreme as the sample suggested, which is the
  argument for excluding them from v1.

## 4. Duplication - how much label text is repeated?  ->  feeds dedup policy

Generic manufacturers and repackagers ship the **same label text** under different NDCs.
`text_hash` is a hash of the section text, so identical text collides regardless of which
record it came from.

This is the failure mode most specific to this corpus: without deduplication the top 5
retrieval results fill with five manufacturers' copies of one paragraph, and the user
sees one answer repeated instead of five pieces of evidence.

Note this measures **exact** duplication only. Near-duplicates (one word changed) escape
hashing entirely, so whatever comes out is a floor, not the true rate.

In [9]:
con.sql('''
    SELECT count(*)                                        AS section_texts,
           count(DISTINCT text_hash)                       AS distinct_texts,
           round(100.0*(1 - count(DISTINCT text_hash)::DOUBLE/count(*)), 1)
                                                           AS pct_exact_duplicate
    FROM sections
''').show()

┌───────────────┬────────────────┬─────────────────────┐
│ section_texts │ distinct_texts │ pct_exact_duplicate │
│     int64     │     int64      │       double        │
├───────────────┼────────────────┼─────────────────────┤
│       4442377 │        1803438 │                59.4 │
└───────────────┴────────────────┴─────────────────────┘



In [10]:
# Restricted to the sections that will actually be indexed, which is what matters.
con.sql(f'''
    SELECT section,
           count(*)                                        AS texts,
           count(DISTINCT text_hash)                       AS distinct_texts,
           round(100.0*(1 - count(DISTINCT text_hash)::DOUBLE/count(*)), 1)
                                                           AS pct_duplicate,
           max(rep)                                        AS most_repeated
    FROM (SELECT section, text_hash,
                 count(*) OVER (PARTITION BY section, text_hash) AS rep
          FROM sections WHERE section IN {ALLOWLIST})
    GROUP BY 1
    ORDER BY pct_duplicate DESC
''').show(max_rows=20)

┌─────────────────────────────┬────────┬────────────────┬───────────────┬───────────────┐
│           section           │ texts  │ distinct_texts │ pct_duplicate │ most_repeated │
│           varchar           │ int64  │     int64      │    double     │     int64     │
├─────────────────────────────┼────────┼────────────────┼───────────────┼───────────────┤
│ overdosage                  │  83888 │          19501 │          76.8 │           470 │
│ contraindications           │  89299 │          26639 │          70.2 │           836 │
│ warnings                    │ 207792 │          68040 │          67.3 │          6390 │
│ indications_and_usage       │ 251968 │          84902 │          66.3 │          5725 │
│ dosage_and_administration   │ 251405 │          96397 │          61.7 │          6346 │
│ drug_interactions           │  68390 │          27337 │          60.0 │           254 │
│ boxed_warning               │  33009 │          13406 │          59.4 │           227 │
│ adverse_

### Duplication - findings

*TODO after running:*

- Corpus-wide exact-duplicate rate: ____%
- Which sections are worst? (Expect boilerplate sections to be near-totally duplicated,
  and drug-specific ones like `adverse_reactions` to be lower.)
- How many times does the single most-repeated text appear?
- What does the chunk count become after dedup, and does that change the Phase 1 gate?
- Does dedup belong at index time, query time, or both?

## 5. Identity - `set_id` vs `id` vs `product_ndc`  ->  feeds the stable chunk key

This decides what chunks are keyed on, and it is a decision that is expensive to reverse
once embeddings exist.

- **`set_id`** is stable across label revisions. When a manufacturer files an updated
  label, `set_id` stays the same and `id` changes.
- **`id`** is version-specific.
- **`product_ndc`** identifies the marketed product, and one label can cover several.

The build plan requires `set_id`. The trap is that if this snapshot holds exactly one
version per label, the two are 1:1 and the wrong choice stays invisible until the corpus
is refreshed, at which point every chunk id silently duplicates.

In [11]:
con.sql('''
    SELECT count(*)                    AS records,
           count(DISTINCT set_id)      AS distinct_set_id,
           count(DISTINCT doc_id)      AS distinct_id,
           count(DISTINCT product_ndc) AS distinct_product_ndc
    FROM meta
''').show()

┌─────────┬─────────────────┬─────────────┬──────────────────────┐
│ records │ distinct_set_id │ distinct_id │ distinct_product_ndc │
│  int64  │      int64      │    int64    │        int64         │
├─────────┼─────────────────┼─────────────┼──────────────────────┤
│  261258 │          261258 │      261258 │                85431 │
└─────────┴─────────────────┴─────────────┴──────────────────────┘



In [12]:
# If any set_id appears more than once, this snapshot already holds label revisions.
con.sql('''
    SELECT n_versions, count(*) AS n_set_ids
    FROM (SELECT set_id, count(*) AS n_versions FROM meta GROUP BY 1)
    GROUP BY 1 ORDER BY 1
''').show()

con.sql("SELECT version, count(*) AS n FROM meta GROUP BY 1 ORDER BY n DESC LIMIT 10").show()

┌────────────┬───────────┐
│ n_versions │ n_set_ids │
│   int64    │   int64   │
├────────────┼───────────┤
│          1 │    261258 │
└────────────┴───────────┘

┌─────────┬───────┐
│ version │   n   │
│ varchar │ int64 │
├─────────┼───────┤
│ 1       │ 62452 │
│ 2       │ 60145 │
│ 3       │ 38981 │
│ 4       │ 26385 │
│ 5       │ 16372 │
│ 6       │ 11160 │
│ 7       │  8292 │
│ 8       │  6055 │
│ 9       │  4606 │
│ 10      │  3506 │
└─────────┴───────┘
      10 rows    



### Identity - findings

*TODO after running:*

- Are `set_id` and `id` 1:1 in this snapshot?
- Does any `set_id` carry more than one version already?
- Confirmed chunk key for Phase 1: ____
- Does one `set_id` map to many `product_ndc`? That decides whether the query-time filter
  keys on label or on product.

## 6. Scope - prescription vs OTC  ->  feeds corpus scope

`HUMAN OTC DRUG` labels are sunscreen, hand sanitiser, cough drops and antacids. They are
real labels, but they are not where drug-safety signal questions live, and they are the
half of the corpus with the trivially short sections seen in section 3.

Combining this with section 2 gives the v1 corpus definition: **`openfda` populated AND
prescription.**

In [13]:
con.sql(f'''
    SELECT coalesce(product_type, '<missing>')          AS product_type,
           count(*)                                     AS records,
           round(100.0*count(*)/{N_RECORDS}, 1)         AS pct
    FROM meta GROUP BY 1 ORDER BY records DESC
''').show()

┌─────────────────────────┬─────────┬────────┐
│      product_type       │ records │  pct   │
│         varchar         │  int64  │ double │
├─────────────────────────┼─────────┼────────┤
│ <missing>               │  174891 │   66.9 │
│ HUMAN OTC DRUG          │   49328 │   18.9 │
│ HUMAN PRESCRIPTION DRUG │   37018 │   14.2 │
│ CELLULAR THERAPY        │      21 │    0.0 │
└─────────────────────────┴─────────┴────────┘



In [14]:
# The v1 corpus, and a rough chunk-count projection for the Phase 1 gate.
scoped = con.sql(f'''
    SELECT count(*) FROM meta
    WHERE has_openfda AND product_type = 'HUMAN PRESCRIPTION DRUG'
''').fetchone()[0]

sec_rows, sec_chars = con.sql(f'''
    SELECT count(*), sum(n_chars) FROM sections s
    JOIN meta m USING (set_id)
    WHERE m.has_openfda AND m.product_type = 'HUMAN PRESCRIPTION DRUG'
      AND s.section IN {ALLOWLIST}
''').fetchone()

print(f"v1 corpus records          {scoped:,}  ({100*scoped/N_RECORDS:.1f}% of {N_RECORDS:,})")
print(f"allowlisted section texts  {sec_rows:,}")
print(f"total characters           {sec_chars:,}")
print(f"projected chunks @400 tok  {sec_chars/(400*4):,.0f}  (before dedup)")

v1 corpus records          37,018  (14.2% of 261,258)
allowlisted section texts  270,901
total characters           1,039,298,978
projected chunks @400 tok  649,562  (before dedup)


### Scope - findings

*TODO after running:*

- Prescription share of the corpus: ____%
- v1 corpus size: ____ records
- Projected chunk count before dedup: ____ ; after applying the section 4 duplicate rate: ____
- Does that land inside the Phase 1 gate of 150k to 700k chunks? If it is far above,
  the scope filter needs tightening before anything is embedded.

## 7. Joinability - can a label reach the marts?  ->  feeds the query-time filter

Chunks are **never** keyed on `drug_key` (ADR-005 salt-stripping bug, needs a live
warehouse to fix). Instead, drug identity is resolved at **query time**, so this measures
whether that resolution can actually succeed.

Two candidate join paths, and the trade between them is the point:

- **`rxcui`** is a numeric code with no typo surface and no salt-stripping. Clean, but
  `int_drug_resolution` carries it on only 4,843 of 84,038 signatures.
- **`canonical_drug_name`** covers far more, and is *also* the field carrying the
  salt-stripping bug (`PHENYTOIN CALCIUM` becomes `PHENYTOIN`). Wide but compromised.

The normalisation below is deliberately crude, uppercase and whitespace only. It sizes the
opportunity; the real join must use `de_capstone`'s `normalize_drug_name`.

In [15]:
con.execute(f"CREATE VIEW res AS SELECT * FROM '{OFFLINE}/int_drug_resolution.parquet'")

con.sql('''
    SELECT count(*)                                  AS signatures,
           count(*) FILTER (WHERE is_resolved)       AS resolved,
           count(rxcui)                              AS with_rxcui,
           count(DISTINCT canonical_drug_name)       AS distinct_canonical
    FROM res
''').show()

┌────────────┬──────────┬────────────┬────────────────────┐
│ signatures │ resolved │ with_rxcui │ distinct_canonical │
│   int64    │  int64   │   int64    │       int64        │
├────────────┼──────────┼────────────┼────────────────────┤
│      84038 │     8443 │       4843 │               4367 │
└────────────┴──────────┴────────────┴────────────────────┘



In [16]:
con.sql(r'''
    WITH scoped AS (
        SELECT DISTINCT
               rxcui,
               upper(trim(regexp_replace(generic_name, '\s+', ' ', 'g'))) AS gen,
               upper(trim(regexp_replace(brand_name,   '\s+', ' ', 'g'))) AS brd
        FROM meta
        WHERE has_openfda AND product_type = 'HUMAN PRESCRIPTION DRUG'
    )
    SELECT
      (SELECT count(DISTINCT s.rxcui) FROM scoped s
         JOIN res r ON s.rxcui = r.rxcui)                                  AS match_rxcui,
      (SELECT count(DISTINCT s.gen)   FROM scoped s
         JOIN res r ON s.gen = upper(trim(r.canonical_drug_name)))         AS match_generic,
      (SELECT count(DISTINCT s.brd)   FROM scoped s
         JOIN res r ON s.brd = upper(trim(r.brand_name)))                  AS match_brand,
      (SELECT count(DISTINCT rxcui) FROM scoped WHERE rxcui IS NOT NULL)   AS label_rxcui,
      (SELECT count(DISTINCT gen)   FROM scoped WHERE gen   IS NOT NULL)   AS label_generic,
      (SELECT count(DISTINCT brd)   FROM scoped WHERE brd   IS NOT NULL)   AS label_brand
''').show()

┌─────────────┬───────────────┬─────────────┬─────────────┬───────────────┬─────────────┐
│ match_rxcui │ match_generic │ match_brand │ label_rxcui │ label_generic │ label_brand │
│    int64    │     int64     │    int64    │    int64    │     int64     │    int64    │
├─────────────┼───────────────┼─────────────┼─────────────┼───────────────┼─────────────┤
│        2673 │          1506 │        3447 │        5000 │          3640 │        5406 │
└─────────────┴───────────────┴─────────────┴─────────────┴───────────────┴─────────────┘



### Joinability - findings

*TODO after running:*

- Which join path reaches more of the corpus, `rxcui` or `canonical_drug_name`?
- Is the `rxcui` path wide enough to use alone, or is a tiered fallback needed (the same
  shape as ADR-005's own resolution ladder)?
- If the canonical-name path is required, note explicitly that it inherits the
  salt-stripping bug, and that this is survivable **because the join happens at query
  time**: fixing ADR-005 later invalidates a join, not the embeddings or the gold set.
- What does a query-time filter miss look like, and what should the fallback be when a
  drug cannot be resolved?

## 8. Scope revisited - was excluding OTC right?  ->  feeds corpus scope

The `HUMAN PRESCRIPTION DRUG` filter in section 6 was a judgment call made before any
data was seen, and sections 1 to 7 carried it forward without testing it. This section
tests it.

### The price of including OTC

In [19]:
ALLOW = ('boxed_warning','adverse_reactions','warnings','warnings_and_cautions',
         'contraindications','drug_interactions','indications_and_usage',
         'use_in_specific_populations','dosage_and_administration','overdosage')

# 1. What does including OTC actually cost?
con.sql(f'''
    SELECT coalesce(m.product_type,'<none>')  AS product_type,
           count(DISTINCT m.set_id)           AS labels,
           count(*)                           AS section_texts,
           sum(s.n_chars)                     AS chars,
           round(sum(s.n_chars)/1600.0)::INT  AS projected_chunks
    FROM sections s JOIN meta m USING (set_id)
    WHERE m.has_openfda AND s.section IN {ALLOW}
    GROUP BY 1 ORDER BY chars DESC
''').show()

┌─────────────────────────┬────────┬───────────────┬────────────┬──────────────────┐
│      product_type       │ labels │ section_texts │   chars    │ projected_chunks │
│         varchar         │ int64  │     int64     │   int128   │      int32       │
├─────────────────────────┼────────┼───────────────┼────────────┼──────────────────┤
│ HUMAN PRESCRIPTION DRUG │  35877 │        270901 │ 1039298978 │           649562 │
│ HUMAN OTC DRUG          │  49328 │        150772 │   49455090 │            30909 │
│ CELLULAR THERAPY        │     21 │           154 │     767975 │              480 │
└─────────────────────────┴────────┴───────────────┴────────────┴──────────────────┘



### Is the allowlist Rx-shaped?

If OTC labels carry their safety content in fields the allowlist does not name, then
including OTC labels would add labels whose safety text is then ignored.

In [20]:
# 2. Is my allowlist Rx-shaped? Which sections does each label type actually use?
con.sql('''
    SELECT s.section,
           count(*) FILTER (WHERE m.product_type='HUMAN PRESCRIPTION DRUG') AS rx,
           count(*) FILTER (WHERE m.product_type='HUMAN OTC DRUG')          AS otc
    FROM sections s JOIN meta m USING (set_id)
    WHERE m.has_openfda
    GROUP BY 1 ORDER BY rx+otc DESC LIMIT 25
''').show(max_rows=25)

┌────────────────────────────────────────────────────────────┬───────┬───────┐
│                          section                           │  rx   │  otc  │
│                          varchar                           │ int64 │ int64 │
├────────────────────────────────────────────────────────────┼───────┼───────┤
│ spl_product_data_elements                                  │ 37018 │ 49328 │
│ package_label_principal_display_panel                      │ 37016 │ 49328 │
│ indications_and_usage                                      │ 35701 │ 49325 │
│ dosage_and_administration                                  │ 35695 │ 49326 │
│ warnings                                                   │ 13626 │ 49315 │
│ inactive_ingredient                                        │   474 │ 49312 │
│ purpose                                                    │   232 │ 49311 │
│ keep_out_of_reach_of_children                              │   243 │ 49297 │
│ active_ingredient                                 

In [21]:
con.sql('''
    SELECT s.section,
           count(*) FILTER (WHERE m.product_type='HUMAN PRESCRIPTION DRUG') AS rx,
           count(*) FILTER (WHERE m.product_type='HUMAN OTC DRUG')          AS otc,
           round(avg(s.n_chars))::INT                                       AS avg_chars
    FROM sections s JOIN meta m USING (set_id)
    WHERE m.has_openfda
      AND s.section IN ('do_not_use','ask_doctor','ask_doctor_or_pharmacist',
                        'stop_use','when_using','warnings','other_safety_information')
    GROUP BY 1 ORDER BY otc DESC
''').show()

┌──────────────────────────┬───────┬───────┬───────────┐
│         section          │  rx   │  otc  │ avg_chars │
│         varchar          │ int64 │ int64 │   int32   │
├──────────────────────────┼───────┼───────┼───────────┤
│ warnings                 │ 13626 │ 49315 │      1456 │
│ stop_use                 │   169 │ 29507 │       172 │
│ when_using               │    59 │ 23173 │       157 │
│ do_not_use               │    69 │ 22453 │       169 │
│ ask_doctor               │    45 │ 17155 │       184 │
│ other_safety_information │    99 │ 10908 │       124 │
│ ask_doctor_or_pharmacist │     5 │  9009 │       140 │
└──────────────────────────┴───────┴───────┴───────────┘



### Scope revisited - findings

Unlike the findings cells above, these are answered: the decision is made.

- **OTC costs 4.8%.** 49,328 OTC labels carry 21x less text than 35,877 prescription ones,
  adding 30,909 chunks on top of 649,562. The filter was excluding 58% of the attributable
  corpus to save 4.8% of the compute.
- **And it removed the wrong drugs.** Acetaminophen, ibuprofen, aspirin, naproxen and
  diphenhydramine are OTC, are among the most-reported substances in FAERS, and are already
  in the marts. BR-16 says nothing about prescription-only.
- **The allowlist was Rx-shaped.** `warnings` was in it by luck. Four more genuine OTC safety
  fields were being discarded: `stop_use` (29,507), `when_using` (23,173), `do_not_use`
  (22,453), `ask_doctor` (17,155), plus `ask_doctor_or_pharmacist` (9,009) and
  `other_safety_information` (10,908).
- **Those fields are tiny**, 124 to 184 characters each, roughly 30 to 46 tokens, six per
  label. Embedded separately they are useless; concatenated they are the Drug Facts safety
  panel, about 900 characters.
- **So the corpus has two opposite problems.** Rx sections are few and enormous and need
  splitting. OTC sections are many and tiny and need merging. One rule solves both: target
  350-450 tokens, split what is over, merge what is under.
- **1,141 prescription labels** carry none of the allowlisted sections (35,877 here vs 37,018
  in section 6) and would produce no chunks at all.

**Decision:** scope filter becomes `record.get("openfda")` non-empty, with no product-type
condition. Cellular therapy stays in: 21 labels, 480 chunks, and CAR-T products carry serious
adverse events. Allowlist goes from 10 sections to 16.

### What about the records with no metadata at all?

Sections 2 and 6 established that 174,891 records (66.9%) have an empty `openfda` block and
are therefore excluded. That is a large exclusion, so it needs a reason stronger than "the
metadata is missing".

The question is whether those records are genuinely unidentifiable, or merely
**unharmonised**. An empty `openfda` block means openFDA's own matching against the NDC
directory failed. It does not mean the label has no drug name on it.

In [22]:
con.sql('''
    WITH nometa AS (SELECT set_id FROM meta WHERE NOT has_openfda)
    SELECT s.section,
           count(*)                        AS labels,
           round(100.0*count(*)/174891, 1) AS pct_of_excluded,
           round(avg(s.n_chars))::INT      AS avg_chars
    FROM sections s JOIN nometa n USING (set_id)
    WHERE s.section IN ('spl_product_data_elements','active_ingredient',
                        'package_label_principal_display_panel','description',
                        'purpose','indications_and_usage','warnings')
    GROUP BY 1 ORDER BY labels DESC
''').show()

┌───────────────────────────────────────┬────────┬─────────────────┬───────────┐
│                section                │ labels │ pct_of_excluded │ avg_chars │
│                varchar                │ int64  │     double      │   int32   │
├───────────────────────────────────────┼────────┼─────────────────┼───────────┤
│ spl_product_data_elements             │ 174593 │            99.8 │       337 │
│ package_label_principal_display_panel │ 174533 │            99.8 │       236 │
│ indications_and_usage                 │ 166921 │            95.4 │       582 │
│ warnings                              │ 144851 │            82.8 │      1311 │
│ purpose                               │ 113806 │            65.1 │        55 │
│ active_ingredient                     │ 111223 │            63.6 │        94 │
│ description                           │  57983 │            33.2 │      1107 │
└───────────────────────────────────────┴────────┴─────────────────┴───────────┘



### Excluded records - findings

**They are not anonymous.** 99.8% carry a product name in `spl_product_data_elements` and
63.6% carry an `active_ingredient` field averaging 94 characters, something like
"Acetaminophen 500 mg". `purpose` on 65.1% identifies what they mostly are: the long tail of
OTC monograph products, store brands and imports that never got harmonised into the NDC
directory.

**They carry real safety content.** 144,851 of them have a `warnings` section averaging
1,311 characters, roughly **190 million characters** of warnings text, about four times the
entire OTC allowlisted corpus. This is not a rounding error.

**Budget is not the reason to exclude them.** Including them adds roughly 180,000 chunks
pre-dedup, taking the projection from 692,000 to about 872,000. That is over the raw ceiling
but lands near 354,000 post-dedup, comfortably inside the gate that actually matters.

**Attribution is the reason, and it is sufficient.** BR-16 is "answer questions about label
content, citing the source". Every query flow starts by resolving a drug and filtering to it.
A chunk that cannot be attached to a drug name cannot be filtered to, cannot be cited
usefully, and if it surfaces anyway it looks like an answer while being unusable.

**Decision: excluded from v1, deferred rather than dismissed.**

> 174,891 labels (66.9%) are excluded because openFDA could not harmonise them to the NDC
> directory. They are not unidentifiable: 99.8% carry a product name in
> `spl_product_data_elements` and 63.6% carry `active_ingredient`. Recovering them means
> resolving free-text ingredient names, which is the ADR-005 problem on a **cleaner input**
> than ADR-005 had: `active_ingredient` is a structured regulatory field, not a human-typed
> FAERS product name, so its resolution rate should beat the 10% signature figure. Deferred
> to a later phase. It would add roughly 190M characters of warnings text.

## Summary - what exploration found, and what it means for chunking

*TODO after running all sections.*

| # | Finding | Consequence for Phase 1 |
|---|---|---|
| 1 | Structure: | |
| 2 | `openfda` coverage: | |
| 3 | Section lengths: | |
| 4 | Duplication: | |
| 5 | Identity: | |
| 6 | Scope: | |
| 7 | Joinability: | |

**v1 corpus definition:** ____

**Projected chunk count:** ____  (Phase 1 gate is 150k to 700k)

**Decisions confirmed or overturned:**

- TR-50 (one chunk per section):
- Chunk key (`set_id`):
- Section allowlist:
- Dedup policy:
- Query-time join path:

Anything here that contradicts `rag/notes/PLAN.md` wins: the plan was drafted from
samples, and this is the full pass.